# Simple RAG Training Pipeline (Cgft)

This notebook follows the same 4-step flow and uses the new CgftPipeline API:

1. **Point to dataset** - Chunk and upload documents
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Load search environment
4. **Run training job** - Launch the training


## Setup

Install dependencies and configure API credentials.


In [1]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev]

In [2]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent]
repo_root = next((p for p in candidate_roots if (p / "src" / "synthetic_data_prep").exists()), cwd)
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [3]:
from synthetic_data_prep.utils import ensure_safe_python_version

ensure_safe_python_version()

In [9]:
# Configuration

# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = "sk_RZEmogqiKirhhDCtipwhSKeqFqnuLUjoxfTQqTkRvNWyuMFOoJkEJxWgJnwoobKS"
BASE_URL = "https://app.cgft.io"
LLM_API_KEY = "EldWZtATVrvUQMKTjgAQOHeadaoiHJoD8SrorScBm6IUn44yARevJQQJ99CBACHYHv6XJ3w3AAAAACOGUUWL"
LLM_BASE_URL = "https://expt-platform-foundry.openai.azure.com/openai/v1"

# Dataset configuration
DOCS_PATH = "./samples/posthog/"
CORPUS_NAME = "posthog"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration (CgftPipeline)
TOTAL_SAMPLES = 10
OUTPUT_DIR = "outputs/cgft_simple_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = "posthog search v1"

# Generation/filter/refinement behavior follows library defaults.
# Override in the config cell below only when you intentionally want non-default behavior.

## Step 1: Point to Dataset

Chunk your documents and upload to corpus API in one line.


In [5]:
from synthetic_data_prep.corpus.corpora.source import CorporaChunkSource

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=BASE_URL)
source.populate_from_folder(DOCS_PATH)

Chunking documents from ./samples/posthog/...
ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3

Uploading chunks:   0%|          | 0/33 [00:00<?, ?batch/s]


Upload complete! Inserted: 3225


## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the corpus from Step 1.


In [13]:
from synthetic_data_prep.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    EntityExtractionLLMConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LinkerConfig,
    LLMDirectGenerationConfig,
    LLMEnvGenerationConfig,
    LLMGuidedLinkerConfig,
    OutputConfig,
    PlatformConfig,
    RefinementConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
    TransformationConfig,
)
from synthetic_data_prep.qa_generation.cgft_pipeline import CgftPipeline

LLM_MODEL = "grok-4-1-fast-non-reasoning"

# --- Batch / concurrency settings ---
# max_concurrent controls how many LLM calls run in parallel within each stage.
# batch_enabled=True  → parallel LLM calls (default, recommended for > ~5 samples)
# batch_enabled=False → sequential calls (useful for debugging a single request)
MAX_CONCURRENT = 8

cfg = CgftPipelineConfig(
    platform=PlatformConfig(api_key=API_KEY, base_url=BASE_URL, llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL),
    corpus=CorpusConfig(corpus_name=CORPUS_NAME, min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        model=LLM_MODEL,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
        # generate_entity_patterns=True (default): LLM generates entity/code/domain patterns
        # from corpus samples during profiling, improving BM25 chunk linking quality.
        # Set to False to skip the extra LLM call and fall back to metadata-based BM25 queries.
        entity_extraction_llm=EntityExtractionLLMConfig(model=LLM_MODEL)
    ),
    linker=LinkerConfig(
        llm_guided=LLMGuidedLinkerConfig(model=LLM_MODEL),
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(
            model=LLM_MODEL,
            # Batch: all generation LLM calls run concurrently (up to MAX_CONCURRENT at a time).
            # Each task gets its own per-prompt system prompt, so per-item context is preserved.
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
            show_batch_progress=True,
        ),
        llm_env=LLMEnvGenerationConfig(model=LLM_MODEL),
    ),
    transformation=TransformationConfig(model=LLM_MODEL, validation_model=LLM_MODEL),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(
            judge_model=LLM_MODEL,
            # Batch: search stays sequential (BM25 per-item), judge calls run concurrently.
            # Items that exceed the overlap pre-gate threshold skip the judge entirely.
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
        grounding_llm=GroundingLLMFilterConfig(
            judge_model=LLM_MODEL,
            # Batch: search stays sequential, grounding judge calls run concurrently.
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
    ),
    refinement=RefinementConfig(model=LLM_MODEL),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)
print("max_same_seed_attempts_before_reanchor:", cfg.refinement.max_same_seed_attempts_before_reanchor)
print("filter chain:", cfg.filtering.filters)
print("generate_entity_patterns:", cfg.corpus_context.generate_entity_patterns)
print()
print("Batch settings")
print(f"  generation  → batch_enabled={cfg.generation.llm_direct.batch_enabled}, max_concurrent={cfg.generation.llm_direct.max_concurrent}, show_progress={cfg.generation.llm_direct.show_batch_progress}")
print(f"  grounding   → batch_enabled={cfg.filtering.grounding_llm.batch_enabled}, max_concurrent={cfg.filtering.grounding_llm.max_concurrent}")
print(f"  retrieval   → batch_enabled={cfg.filtering.retrieval_llm.batch_enabled}, max_concurrent={cfg.filtering.retrieval_llm.max_concurrent}")

generator model: grok-4-1-fast-non-reasoning
max_same_seed_attempts_before_reanchor: 3
filter chain: ['grounding_llm', 'retrieval_too_easy_llm']
generate_entity_patterns: True

Batch settings
  generation  → batch_enabled=True, max_concurrent=8, show_progress=True
  grounding   → batch_enabled=True, max_concurrent=8
  retrieval   → batch_enabled=True, max_concurrent=8


In [14]:
from synthetic_data_prep.qa_generation.generators.direct_llm import _DEFAULT_TEMPLATE
from synthetic_data_prep.qa_generation.linkers import (
    RELATED_CHUNK_SYSTEM_PROMPT,
    RELATED_CHUNK_USER_TEMPLATE,
)


def _print_prompt_block(title: str, text: str) -> None:
    print(f"\n{'=' * 16} {title} {'=' * 16}")
    print((text or "(empty)").strip() or "(empty)")


seed_corpus_summary = str(cfg.corpus_context.description or "").strip()
seed_query_list = [str(q).strip() for q in (cfg.corpus_context.example_queries or []) if str(q).strip()]
seed_corpus_queries = "\n".join(f"- {q}" for q in seed_query_list)

print("Corpus context to prompt variables")
_print_prompt_block("Seed corpus_summary (from description)", seed_corpus_summary)
_print_prompt_block("Seed corpus_queries (from example_queries)", seed_corpus_queries)
print(f"\nCorpus profiling enabled: {cfg.corpus_context.enabled}")
print(f"Entity pattern generation enabled: {cfg.corpus_context.generate_entity_patterns}")
print("If profiling succeeds, corpus_summary/corpus_queries and entity patterns are generated before QA generation.")


print("\nActive pipeline prompt templates")

_print_prompt_block("Corpus profile system prompt", cfg.corpus_context.system_prompt)
_print_prompt_block("Corpus profile user template", cfg.corpus_context.user_template)

if cfg.linker.type == "llm_guided":
    linker_system = cfg.linker.llm_guided.system_prompt or RELATED_CHUNK_SYSTEM_PROMPT
    linker_user = cfg.linker.llm_guided.user_template or RELATED_CHUNK_USER_TEMPLATE
    _print_prompt_block("LLM-guided linker system prompt", linker_system)
    _print_prompt_block("LLM-guided linker user template", linker_user)
else:
    print("\nLinker mode: structural (entity patterns used for BM25 if generate_entity_patterns=True).")

if cfg.generation.mode == "llm_direct":
    _print_prompt_block("Direct generation system prompt", cfg.generation.llm_direct.system_prompt)
    templates = cfg.generation.llm_direct.prompt_templates_by_qa_type
    if templates:
        for qa_type in sorted(templates):
            _print_prompt_block(f"Direct generation template [{qa_type}]", templates[qa_type])
    else:
        _print_prompt_block("Direct generation default template", _DEFAULT_TEMPLATE)
else:
    _print_prompt_block("Env generation prompt template", cfg.generation.llm_env.prompt_template)

active_filter_chain = [
    str(name).strip().lower()
    for name in (cfg.filtering.filters)
    if str(name).strip()
]
print("Active filter chain:", active_filter_chain)

if "retrieval_llm" in active_filter_chain or "retrieval_too_easy_llm" in active_filter_chain:
    _print_prompt_block("Retrieval judge system prompt", cfg.filtering.retrieval_llm.judge_system_prompt)
    _print_prompt_block("Retrieval judge user template", cfg.filtering.retrieval_llm.judge_user_template)

if "grounding_llm" in active_filter_chain:
    _print_prompt_block("Grounding judge system prompt", cfg.filtering.grounding_llm.judge_system_prompt)
    _print_prompt_block("Grounding judge user template", cfg.filtering.grounding_llm.judge_user_template)

if "llm_env" in active_filter_chain:
    from synthetic_data_prep.qa_generation.filters.env_rollout import _JUDGE_SYSTEM, _JUDGE_USER

    _print_prompt_block(
        "Env filter rollout prompt",
        "Answer the question with tool use as needed. Return final answer in <answer>...</answer>.\n\nQuestion: {question}",
    )
    _print_prompt_block("Env filter judge system prompt", _JUDGE_SYSTEM)
    _print_prompt_block("Env filter judge user template", _JUDGE_USER)

if cfg.refinement.enabled:
    _print_prompt_block("Refinement prompt template", cfg.refinement.prompt_template)
else:
    print("\nRefinement disabled.")

print("\nTransformation settings")
print(
  {
      "noise_level": cfg.transformation.noise_level,
      "validation_enabled": cfg.transformation.validation_enabled,
      "validation_model": cfg.transformation.validation_model
  }

)
_print_prompt_block("Transformation system prompt", cfg.transformation.system_prompt)
_print_prompt_block("Transformation user template", cfg.transformation.user_template)


Corpus context to prompt variables

================ Seed corpus_summary (from description) ================
Posthog documentation including docs, company policy, etc.

================ Seed corpus_queries (from example_queries) ================
- how to feature flag
- setup reverse proxy to avoid ad blockers
- posthog install nextjs

Corpus profiling enabled: True
Entity pattern generation enabled: True
If profiling succeeds, corpus_summary/corpus_queries and entity patterns are generated before QA generation.

Active pipeline prompt templates

================ Corpus profile system prompt ================
You are a technical analyst specializing in document corpus analysis.
Your goal is to understand the overall themes, content type, and typical query that a user might search for.

Based on the context and thoughts, form an understanding of the corpus, and then provide a short summary and example search queries. Weigh the user's input if provided.

Guideline:
- Try to generalize and 

### Batch / parallel processing

With `batch_enabled=True` (the default), LLM calls within each pipeline stage run concurrently up to `max_concurrent`. Stages themselves still run in order:

```
Generator.generate(tasks)
    ├─ prepare all tasks sequentially (linker.link per task)
    └─ batch LLM call  ← parallel, up to max_concurrent

GroundingFilter.evaluate(items)
    ├─ BM25 search sequentially (one per item)
    └─ batch judge calls  ← parallel, up to max_concurrent

RetrievalFilter.evaluate(items)
    ├─ BM25 search sequentially
    ├─ items that hit the overlap pre-gate → verdict immediately (no judge call)
    └─ remaining items → batch judge calls  ← parallel
```

Expected speedup vs sequential (`max_concurrent=1`): **3–6× faster** for batches of ≥ 20 items, limited by LLM latency rather than call count.

To debug a single item, set `batch_enabled=False` in any of the three configs above.

In [15]:
# Reuse the already-loaded corpus source from Step 1.
import importlib
from pprint import pprint

import synthetic_data_prep.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import synthetic_data_prep.qa_generation.generators.direct_llm as _direct_llm_mod

# Force-refresh generation modules in case notebook state still has stale classes.
importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]

# Show entity extraction patterns generated during corpus profiling.
# These drive BM25 query generation in the structural linker, replacing brittle
# metadata-based queries (headers/first sentence) with corpus-specific entities.
entity_extraction = cfg.corpus_context.entity_extraction
if entity_extraction is not None:
    print("Entity extraction patterns (confidence=%s)" % entity_extraction.confidence)
    pprint(
        {
            "entity_names": entity_extraction.entity_names,
            "domain_terms": entity_extraction.domain_terms,
            "code_patterns": entity_extraction.code_patterns,
            "query_templates": entity_extraction.query_templates,
        }
    )
else:
    print("Entity extraction patterns: not generated (generate_entity_patterns=False or profiling disabled)")

print()
print("Run summary")
pprint(
    {
        "raw_candidates_total": stats.get("raw_candidates_total"),
        "passed_total": stats.get("passed_total"),
        "rejected_total": stats.get("rejected_total"),
        "regenerated_total": stats.get("regenerated_total"),
        "round_limit": stats.get("round_limit"),
        "filter_chain": stats.get("filter_chain"),
    }
)

retrieval_stats = stats.get("retrieval_too_easy_filter") or {}
if retrieval_stats:
    print()
    print("Retrieval filter diagnostics")
    pprint(
        {
            "passed": retrieval_stats.get("passed"),
            "needs_refinement": retrieval_stats.get("needs_refinement"),
            "rejected": retrieval_stats.get("rejected"),
            "errors": retrieval_stats.get("errors"),
            "judge_calls": retrieval_stats.get("judge_calls"),
            "judge_answerable_true": retrieval_stats.get("judge_answerable_true"),
            "judge_answerable_false": retrieval_stats.get("judge_answerable_false"),
            "too_easy_total": retrieval_stats.get("too_easy_total"),
            "too_easy_due_to_overlap_pre_gate": retrieval_stats.get("too_easy_due_to_overlap_pre_gate"),
            "too_easy_due_to_judge": retrieval_stats.get("too_easy_due_to_judge"),
            "overlap_threshold_triggered": retrieval_stats.get("overlap_threshold_triggered"),
            "too_easy_overlap_threshold_triggered": retrieval_stats.get("too_easy_overlap_threshold_triggered"),
        }
    )
    print("judge_reason_tags:")
    pprint(retrieval_stats.get("judge_reason_tags", {}))
else:
    print()
    print("Retrieval (too-easy) filter diagnostics are unavailable in result['stats'].")

grounding_stats = stats.get("grounding_filter", {})
if grounding_stats:
    print()
    print("Grounding filter diagnostics")
    pprint(
        {
            "passed": grounding_stats.get("passed"),
            "needs_refinement": grounding_stats.get("needs_refinement"),
            "rejected": grounding_stats.get("rejected"),
            "errors": grounding_stats.get("errors"),
            "judge_answerable_true": grounding_stats.get("judge_answerable_true"),
            "judge_answerable_false": grounding_stats.get("judge_answerable_false"),
            "unsupported_total": grounding_stats.get("unsupported_total"),
        }
    )

transformation_stats = stats.get("transformation", {})
if transformation_stats:
    print()
    print("Transformation diagnostics")
    pprint(transformation_stats)

stats


Processing prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Entity extraction patterns (confidence=high)
{'code_patterns': {'config_keys': '(api_host|ui_host|host|privacy_mode|feature_flag_request_timeout_ms|privacyMode|posthog_privacy_mode|posthogPrivacyMode)',
                   'domains': '(us\\.i\\.posthog\\.com|eu\\.i\\.posthog\\.com|us-assets\\.i\\.posthog\\.com|eu-assets\\.i\\.posthog\\.com|us\\.posthog\\.com|eu\\.posthog\\.com)',
                   'file_extensions': '(\\.parquet|\\.zst|\\.json)',
                   'file_paths': '(proxy\\.js|middleware\\.js|theme\\.liquid|manifest\\.json)',
                   'function_calls': '(posthog\\.init|isFeatureEnabled|client\\.responses\\.create|client\\.messages\\.create|client\\.models\\.generate_content)',
                   'headers': '(Content-Security-Policy|Access-Control-Allow-Origin|frame-ancestors)',
                   'proxy_paths': '(\\/ph|^\\/ph)'},
 'domain_terms': ['feature flags',
                  'correlation analysis',
                  'funnel insight',
                  'c

{'total': 9,
 'train': 4,
 'eval': 5,
 'train_ratio': 0.8,
 'stratify_by': ['qa_type', 'style'],
 'raw_candidates_total': 10,
 'passed_total': 9,
 'rejected_total': 1,
 'regenerated_total': 2,
 'round_limit': 4,
 'filter_chain': ['grounding_llm', 'retrieval_too_easy_llm'],
 'deterministic_guards': {'passed': 12, 'rejected': 0},
 'retrieval_too_easy_filter': {'passed': 9,
  'needs_refinement': 0,
  'rejected': 0,
  'errors': 0,
  'judge_calls': 9,
  'judge_answerable_true': 2,
  'judge_answerable_false': 7,
  'judge_reason_tags': {'too_easy_lexical': 0,
   'unsupported': 0,
   'challenging_answerable_pass': 0,
   'unknown': 9},
  'too_easy_total': 0,
  'too_easy_due_to_overlap_pre_gate': 0,
  'too_easy_due_to_judge': 0,
  'overlap_threshold_triggered': 1,
  'too_easy_overlap_threshold_triggered': 0},
 'grounding_filter': {'passed': 9,
  'needs_refinement': 2,
  'rejected': 1,
  'errors': 0,
  'judge_answerable_true': 9,
  'judge_answerable_false': 3,
  'unsupported_total': 3},
 'transfo

In [16]:
result

{'train_dataset': [{'task_id': 'task_00002',
   'question': 'PostHog web SDK FF $initial_li_fat_id example value privacy $la_fat_id',
   'answer': 'The web SDK sends `$initial_li_fat_id: testLiFatId123` for the first-page LinkedIn Ad Tracking ID in feature flag requests. In privacy/data storage docs, the equivalent LinkedIn property is listed as `$la_fat_id` with example `testLaFatId123`.',
   'qa_type': 'cross_document_multi_hop',
   'style_target': 'keyword',
   'style_observed': 'natural',
   'min_hop_count': 3,
   'reference_chunks': [{'id': 'd6f7837124e66cb7d7029ee928870d5138cd01ddf02f8d750b986ae2851e4ba0',
     'metadata': {'h2': 'Processing data before storage',
      'h3': 'Discarding IP addresses at project level',
      'index': 4,
      'file': 'privacy/data-storage.mdx'},
     'content': "| Facebook Click ID           | `$fbclid`                | `testFbclid123`                |\n| Microsoft Click ID          | `$msclkid`               | `testMsclkid123`               |\n| 

## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import synthetic_data_prep
from synthetic_data_prep.envs.cgft_search_env import CgftSearchEnv
from synthetic_data_prep.trainer.pipeline import train
experiment_id = train(
    env_class=CgftSearchEnv,
    env_args={
        "api_key": API_KEY,
        "corpus_id": source.corpus_id,
        "base_url": BASE_URL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="cgft-search",
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[synthetic_data_prep],
    experiment_name=EXPERIMENT_NAME,
)


## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse
